<a href="https://colab.research.google.com/github/CarlKo-DLSU/PowerpuffCarl-MP1/blob/main/MP_Problem2_PowerpuffCarl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MP1 - Problem 2 | Powerpuff Carl

This script demonstrates how to build *reversible* quantum circuits for several
3-input classical logic gates:

- 3-qubit XNOR : output = NOT(a XOR b XOR c)
- 3-qubit NAND : output = NOT(a AND b AND c)
- 3-qubit OR   : output = a OR b OR c
- 3-qubit NOR  : output = NOT(a OR b OR c)

Classical logic gates such as AND, OR, XOR are *irreversible* — they lose information.
Quantum circuits must always be *reversible* (unitary), so we must use additional
ancilla qubits or gate constructions (like Toffoli gates) to compute these functions
without destroying input data.

This file:
 1. Defines circuit constructors for reversible versions of classical gates,
 2. Converts classical 0/1 inputs into |0⟩/|1⟩ basis states,
 3. Extracts classical outputs by interpreting the resulting statevector,
 4. Validates the quantum circuits against classical truth tables.

The explanations below describe *why* each part is needed from a quantum computing perspective.

# Install required libraries

Qiskit is used to construct quantum circuits and simulate statevectors.
pylatexenc is used to convert LaTeX-like labels such as "|ψ⟩" into plain text.
numpy is needed for vector calculations, probabilities, and bit operations.

In [ ]:
!pip install qiskit
!pip install pylatexenc
!pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=12c6e61ca3303ac82881a60255818af1d6ff6676beb911a578df99b258bc73af
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


# Imports

QuantumCircuit      → Build quantum circuits composed of quantum gates

Statevector         → Simulate the exact wavefunction of the circuit

numpy               → Numerical operations on amplitudes / bit masks

LatexNodes2Text     → Convert circuit labels from LaTeX ("|0⟩") to plain text

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np
from pylatexenc.latex2text import LatexNodes2Text

# Prepare input qubits into the computational basis

Quantum circuits always begin with all qubits in |0⟩.

To encode classical bits (0 or 1), we apply an X gate (NOT) to flip |0⟩ → |1⟩.
This function:

- Accepts a list of classical bits: bits = (a, b, c)
- Applies X to qubit[i] if bits[i] == 1
- Leaves qubit[i] unchanged if bits[i] == 0

In [ ]:
def prepare_inputs(qc: QuantumCircuit, bits, input_qubits):
    """Prepare computational-basis input states according to bits (iterable of 0/1)."""
    for i, b in enumerate(bits):
        if b:
            qc.x(input_qubits[i])

# Extract classical output from a statevector

After the circuit runs, the simulator gives a statevector (superposition).

Our circuits are fully deterministic:
  • Inputs are classical (|0⟩ or |1⟩)
  • Outputs are produced through Toffoli / CNOT networks (no superpositions)

Thus the target qubit ends in exactly |0⟩ or |1⟩,
and we extract this by summing probabilities of basis states.

In [ ]:
def read_target_from_statevector(sv: Statevector, target_idx: int) -> int:
    """Return deterministic target bit (0/1) by summing probabilities using Qiskit's
    bitstring labels. Qiskit labels are MSB...LSB, so the target qubit's bit is at
    position -1-target_idx in the string (target_idx: 0 = qubit 0)."""
    probs = sv.probabilities_dict()  # e.g. {'000': 1.0, ...}
    p1 = 0.0
    for bitstr, p in probs.items():
        # Qiskit bitstr is ordered MSB...LSB, so target qubit index maps to bitstr[-1-target_idx]
        if bitstr[-1 - target_idx] == "1":
            p1 += p
    return 1 if p1 > 0.5 else 0

# 3-qubit XNOR circuit (reversible)

    XNOR(a,b,c) = NOT(a XOR b XOR c)

This is simply the negation of the parity (XOR) of the inputs.

To compute XOR reversibly:

- CNOT acts as: target ^= control
- So XOR of multiple bits can be built by chaining CNOTs into the target

Procedure:
  1. Encode inputs
  2. Use:
        - t ^= a
        - t ^= b
        - t ^= c

     so target = a⊕b⊕c
  3. Apply X to get XNOR

Circuit layout:
- qubit 0 = a
- qubit 1 = b
- qubit 2 = c
- qubit 3 = target (initialized to |0⟩)

In [ ]:
def build_xnor3_circuit(bits):
    """4-qubit circuit: q0,q1,q2 = inputs; q3 = target. target = XNOR(a,b,c)."""
    qc = QuantumCircuit(4, name="XNOR3")
    inputs = [0, 1, 2]
    target = 3
    prepare_inputs(qc, bits, inputs)
    # Compute parity into target (XOR): target ^= a ^ b ^ c
    qc.cx(0, target)
    qc.cx(1, target)
    qc.cx(2, target)
    # Invert to get XNOR
    qc.x(target)
    return qc

# 3-qubit NAND circuit (reversible)

    NAND(a,b,c) = NOT(a AND b AND c)

Quantum implementation:
- a AND b AND c is computed with Toffoli (CCX) gates.
- Toffoli is the reversible version of classical AND with control bits.

Procedure:
  1. anc = a AND b
  2. target = anc AND c  → target = a AND b AND c
  3. X(target) to compute NAND
  4. uncompute anc = revert ancilla to |0⟩ (mandatory to preserve reversibility)

Uncomputation is required because:
  - Ancillas must be returned to |0⟩ for reversibility (unitarity)
  - Leaving ancillas dirty breaks reversibility and is disallowed in clean logic synthesis

In [ ]:
def build_nand3_circuit(bits):
    """5-qubit circuit: q0,q1,q2 = inputs; q3 = anc; q4 = target.
       target = NAND(a,b,c) = NOT(a AND b AND c)"""
    qc = QuantumCircuit(5, name="NAND3")
    inputs = [0, 1, 2]
    anc = 3
    target = 4
    prepare_inputs(qc, bits, inputs)
    # anc = a & b
    qc.ccx(0, 1, anc)
    # target = anc & c  -> target = a & b & c
    qc.ccx(anc, 2, target)
    # invert target -> NAND
    qc.x(target)
    # uncompute anc to restore anc=0
    qc.ccx(0, 1, anc)
    return qc

# 3-qubit OR circuit (reversible)

    OR(a,b,c) = NOT( AND(NOT a, NOT b, NOT c) )

We use De Morgan's law:

    a OR b OR c = NOT ( (NOT a) AND (NOT b) AND (NOT c) )

This method avoids needing OR gates (irreversible) and expresses OR through AND,
which *can* be implemented reversibly with Toffoli.

Procedure:
  1. Negate inputs: not a, not b, not c
  2. Compute AND(not a, not b, not c) using ancillas
  3. Invert result to obtain OR
  4. Undo all temporary operations (uncompute)

In [ ]:
def build_or3_circuit(bits):
    """5-qubit circuit: q0,q1,q2 = inputs; q3 = anc; q4 = target.
       target = OR(a,b,c) implemented via De Morgan:
       OR = NOT(AND(NOT a, NOT b, NOT c))"""
    qc = QuantumCircuit(5, name="OR3")
    inputs = [0, 1, 2]
    anc = 3
    target = 4
    prepare_inputs(qc, bits, inputs)
    # invert inputs
    qc.x(0)
    qc.x(1)
    qc.x(2)
    # anc = (not a) & (not b)
    qc.ccx(0, 1, anc)
    # target = anc & (not c) -> AND(not a, not b, not c)
    qc.ccx(anc, 2, target)
    # OR = NOT(AND(not a, not b, not c))
    qc.x(target)
    # uncompute anc and restore inputs
    qc.ccx(0, 1, anc)
    qc.x(0)
    qc.x(1)
    qc.x(2)
    return qc

# 3-qubit NOR circuit (reversible)

    NOR(a,b,c) = NOT(a OR b OR c)

Or equivalently,

    NOR(a,b,c) = AND(NOT a, NOT b, NOT c)

Notice this is nearly identical to the previous OR circuit, but without
the final inversion step.

Procedure:
  1. Invert inputs
  2. Compute AND(not a, not b)
  3. Compute AND(previous, not c)
  4. Uncompute ancilla
  5. Restore inputs

We compute AND(NOT a, NOT b, NOT c) directly and output it.

In [ ]:
def build_nor3_circuit(bits):
    """5-qubit circuit: q0,q1,q2 = inputs; q3 = anc; q4 = target.
       target = NOR(a,b,c) = AND(NOT a, NOT b, NOT c)"""
    qc = QuantumCircuit(5, name="NOR3")
    inputs = [0, 1, 2]
    anc = 3
    target = 4
    prepare_inputs(qc, bits, inputs)
    # invert inputs
    qc.x(0)
    qc.x(1)
    qc.x(2)
    # anc = (not a) & (not b)
    qc.ccx(0, 1, anc)
    # target = anc & (not c) -> AND(not a, not b, not c)
    qc.ccx(anc, 2, target)
    # uncompute anc and restore inputs
    qc.ccx(0, 1, anc)
    qc.x(0)
    qc.x(1)
    qc.x(2)
    return qc

# Classical reference functions

These are ordinary Python implementations of XNOR, NAND, OR, NOR.
They are used to verify that the quantum circuits evaluate identical results
for all 8 possible input combinations (0,1)^3.

In [ ]:
# Classical reference implementations for validation
def classical_xnor3(a, b, c):
    return 0 if (a ^ b ^ c) else 1

def classical_nand3(a, b, c):
    return 0 if (a and b and c) else 1

def classical_or3(a, b, c):
    return 1 if (a or b or c) else 0

def classical_nor3(a, b, c):
    return 1 if not (a or b or c) else 0

# Full validation function

validate_all() performs exhaustive validation:

For each gate type:
  - Iterate through all 8 input combinations (a,b,c)
  - Build quantum circuit for those inputs
  - Simulate circuit using Statevector
  - Extract target qubit classical output
  - Compare against classical reference function
  - Record results in a summary dictionary

This ensures that the reversible circuits correctly implement the
intended classical logic functions.

In [ ]:
def validate_all():
    """Validate all four quantum circuits against classical truth tables.
    Prints per-entry comparisons and returns a summary dict."""
    summary = {}
    latex = LatexNodes2Text()
    gates = [
        ("XNOR3", build_xnor3_circuit, classical_xnor3, 4, 3),  # (name, builder, classical_fn, n_qubits, target_idx)
        ("NAND3", build_nand3_circuit, classical_nand3, 5, 4),
        ("OR3", build_or3_circuit, classical_or3, 5, 4),
        ("NOR3", build_nor3_circuit, classical_nor3, 5, 4),
    ]
    for name, builder, cref, _, target in gates:
        correct = True
        table = []
        for a in (0, 1):
            for b in (0, 1):
                for c in (0, 1):
                    bits = (a, b, c)
                    qc = builder(bits)
                    sv = Statevector.from_instruction(qc)
                    out = read_target_from_statevector(sv, target_idx=target)
                    cref_out = cref(a, b, c)
                    ok = (out == cref_out)
                    table.append(((a, b, c), out, cref_out, ok))
                    if not ok:
                        correct = False
        summary[name] = {"valid": correct, "details": table}
    return summary

# Main execution (runs validation)

When executed as a script:
  - Computes truth tables for XNOR, NAND, OR, NOR
  - Compares quantum vs classical outputs
  - Prints detailed results

This ensures full correctness of the reversible quantum gate constructions.

In [ ]:
if __name__ == "__main__":
    summary = validate_all()
    for gate, info in summary.items():
        print(f"{LatexNodes2Text().latex_to_text(gate)}: valid = {info['valid']}")
        print(" truth table (input -> circuit_out, classical, ok):")
        for entry in info["details"]:
            bits, out, cref_out, ok = entry
            print(f"  {bits} -> {out}, {cref_out}, {ok}")
        print()

XNOR3: valid = True
 truth table (input -> circuit_out, classical, ok):
  (0, 0, 0) -> 1, 1, True
  (0, 0, 1) -> 0, 0, True
  (0, 1, 0) -> 0, 0, True
  (0, 1, 1) -> 1, 1, True
  (1, 0, 0) -> 0, 0, True
  (1, 0, 1) -> 1, 1, True
  (1, 1, 0) -> 1, 1, True
  (1, 1, 1) -> 0, 0, True

NAND3: valid = True
 truth table (input -> circuit_out, classical, ok):
  (0, 0, 0) -> 1, 1, True
  (0, 0, 1) -> 1, 1, True
  (0, 1, 0) -> 1, 1, True
  (0, 1, 1) -> 1, 1, True
  (1, 0, 0) -> 1, 1, True
  (1, 0, 1) -> 1, 1, True
  (1, 1, 0) -> 1, 1, True
  (1, 1, 1) -> 0, 0, True

OR3: valid = True
 truth table (input -> circuit_out, classical, ok):
  (0, 0, 0) -> 0, 0, True
  (0, 0, 1) -> 1, 1, True
  (0, 1, 0) -> 1, 1, True
  (0, 1, 1) -> 1, 1, True
  (1, 0, 0) -> 1, 1, True
  (1, 0, 1) -> 1, 1, True
  (1, 1, 0) -> 1, 1, True
  (1, 1, 1) -> 1, 1, True

NOR3: valid = True
 truth table (input -> circuit_out, classical, ok):
  (0, 0, 0) -> 1, 1, True
  (0, 0, 1) -> 0, 0, True
  (0, 1, 0) -> 0, 0, True
  (0, 1, 1

# Results

Executing the program will display a validation summary confirming that all four quantum circuits (XNOR3, NAND3, OR3, NOR3) correctly implement their classical logic counterparts, with detailed truth table comparisons printed for each gate.